# GraphFrames

Dans ce TP, nous allons utiliser la librairie GraphFrame. Il faut adapter légèrement la création de la SparkSession pour l'utiliser (ou l'installer avec pip).

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab7GraphFramesApp") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.5-s_2.12") \
    .getOrCreate()

In [29]:
from functools import reduce

from graphframes import GraphFrame

Il est possible de créer simplement des GraphFrames à partir de DataFrames de sommets et d'arêtes.

<br>- Le DataFrame de sommets doit contenir une colonne spéciale, nommée "id", qui spécifie des identifiants uniques pour chaque sommet dans le graphe.
<br>- Le DataFrame d'arêtes doit contenir deux colonnes spéciales : "src" (identifiant du sommet source de l'arête) et "dst" (identifiant du sommet destination de l'arête).
<br>- Les deux DataFrames peuvent avoir d'autres colonnes arbitraires. Ces colonnes peuvent représenter des attributs de sommets et d'arêtes.

On se donne la liste des personnes suivantes, caractérisée par 3 champs : l'id de la personne (en réalité du futur noeud), son nom et son âge. Ces personnes sont destinées à être les noeuds de notre futur graphe.

In [3]:
people = [
    ("a", "Alice", 34),
    ("b", "Bob", 36),
    ("c", "Charlie", 30),
    ("d", "David", 29),
    ("e", "Esther", 32),
    ("f", "Fanny", 36),
    ("g", "Gabby", 60),
]

Créer un dataframe vertices à partir de cette liste.

In [4]:
vertices = spark.createDataFrame(people, schema=["id", "name", "age"])

In [5]:
vertices.toPandas()

,id,name,age
0,a,Alice,34
1,b,Bob,36
2,c,Charlie,30
3,d,David,29
4,e,Esther,32
5,f,Fanny,36
6,g,Gabby,60


Ces personnes sont inscrites à un réseau social où deux options sont possibles :
- suivre une personne ("follow"),
- déclarer être ami avec cette personne ("friend"), ce qui n'est pas forcément réciproque.
Il n'est pas possible de cumuler les deux status.

Voici l'ensemble des relations :
<br>- Alice a déclaré être amie avec Bob.
<br>- Bob suit Charlie.
<br>- Charlie suit Bob.
<br>- Fanny suit Charlie.
<br>- Esther suit Fanny.
<br>- Esther a déclaré être amie avec David.
<br>- David a déclaré être ami avec Alice.
<br>- Alice a déclaré être amie avec Esther.

Créer un dataframe contenant l'ensemble de ces interactions. Une interaction sera caractérisée par l'identifiant de la personne à l'origine de l'interaction ("src"), celui de la personne visée par l'interaction ("dst"), et le statut de la relation ("relationship", valant "follow" ou "friend").

In [7]:
edges = spark.createDataFrame(
    [
        ("a", "b", "friend"),
        ("b", "c", "follow"),
        ("c", "b", "follow"),
        ("f", "c", "follow"),
        ("e", "f", "follow"),
        ("e", "d", "friend"),
        ("d", "a", "friend"),
        ("a", "e", "friend"),
    ],
    schema=["src", "dst", "relationship"],
)

Créer un graphe représentatif de la situation à partir de ces deux dataframes.

In [8]:
g = GraphFrame(vertices, edges)
print(g)

GraphFrame(v:[id: string, name: string ... 1 more field], e:[src: string, dst: string ... 1 more field])



## Requêtes de base sur les graphes et les DataFrames
Les objets de type GraphFrame fournissent plusieurs méthodes natives pour manipuler les graphes. Nous allons les manipuler ici.

Afficher les sommets/noeuds du graphe, ainsi que ses arcs. Afin de savoir dans quels attributs de notre objet ils sont stockés, consulter la documentation de la classe GraphFrame : https://graphframes.github.io/graphframes/docs/_site/api/python/graphframes.html.

In [10]:
g.vertices.toPandas()

,id,name,age
0,a,Alice,34
1,b,Bob,36
2,c,Charlie,30
3,d,David,29
4,e,Esther,32
5,f,Fanny,36
6,g,Gabby,60


In [11]:
g.edges.toPandas()

,src,dst,relationship
0,a,b,friend
1,b,c,follow
2,c,b,follow
3,f,c,follow
4,e,f,follow
5,e,d,friend
6,d,a,friend
7,a,e,friend


Déterminer le "degré entrant" de l'ensemble des sommets (i.e. pour chaque sommet, le nombre de relations qui pointent vers ce sommet).

In [13]:
g.inDegrees.show()

+---+--------+
| id|inDegree|
+---+--------+
|  b|       2|
|  c|       2|
|  f|       1|
|  d|       1|
|  a|       1|
|  e|       1|
+---+--------+



Déterminer le degré sortant de l'ensemble des sommets.

In [14]:
g.outDegrees.show()

+---+---------+
| id|outDegree|
+---+---------+
|  a|        2|
|  b|        1|
|  c|        1|
|  f|        1|
|  e|        2|
|  d|        1|
+---+---------+



Déterminer le degré des sommets (somme des degrés entrant et sortant).

In [15]:
g.degrees.show()

+---+------+
| id|degree|
+---+------+
|  b|     3|
|  a|     3|
|  c|     3|
|  f|     2|
|  e|     3|
|  d|     2|
+---+------+



Il est possible d'exécuter directement des requêtes sur le DataFrame des sommets.
Trouver l'âge de la personne la plus jeune dans le graphe. Utiliser le DSL.

In [18]:
g.vertices.groupBy().min("age").show()

+--------+
|min(age)|
+--------+
|      29|
+--------+



Il est bien entendu également possible d'exécuter des requêtes sur le DataFrame des arcs.
Compter le nombre de relations de type "follow" dans le graphe.

In [20]:
from pyspark.sql.functions import col
g.edges.filter(col("relationship") == "follow").count()

4

## Trouver des motifs

En utilisant des motifs, il est possible construire des relations plus complexes impliquant des arêtes et des sommets.

<br>Les motifs à rechercher sont dénotés par des expressions.
<br>Un expression élémentaire est "(a)-[e]->(b)". Elle signifie qu'il existe une arête dirigée de a vers b.
<br>Il est possible de combiner ces expressions (le symbole ; est utilisé pour exprimer le ET logique).
<br>Pour les sommets, la répétition d'une lettre signifie que la référence se fait à un même sommet.
<br> Pour les arêtes, il n'est pas possible de répéter la même lettre.
<br>Une fois l'expression construite, il faut appeler la méthode find de la manière suivante : graph.find(motif)

Trouver toutes les paires de sommets avec des arêtes dans les deux directions entre eux (les relations directe et réciproque ne sont pas forcément les mêmes). Le résultat doit être un DataFrame, dans lequel les noms de colonnes sont donnés par les clés du motif.

In [24]:
motif_df = g.find("(a)-[e]->(b); (b)-[f]->(a)")

In [25]:
motif_df.toPandas()

,a,e,b,f
0,"(c, Charlie, 30)","(c, b, follow)","(b, Bob, 36)","(b, c, follow)"
1,"(b, Bob, 36)","(b, c, follow)","(c, Charlie, 30)","(c, b, follow)"


Puisque le résultat est un DataFrame, il est possible d'exécuter des requêtes par dessus le motif.
Parmi les relations précédentes, déterminer celles qui concernent deux personnes dont l'une au moins est âgée de 34 ans ou plus.

In [27]:
motif_df.filter("a.age >= 34 or b.age >= 34").toPandas()

,a,e,b,f
0,"(c, Charlie, 30)","(c, b, follow)","(b, Bob, 36)","(b, c, follow)"
1,"(b, Bob, 36)","(b, c, follow)","(c, Charlie, 30)","(c, b, follow)"



## Requêtes stateful
La plupart des requêtes motif sont sans état et simples à exprimer, comme dans notre exemple précédent. Parfois, une requête plus complexe doit transporter un état le long d'un chemin dans le motif. On peut l'exprimer en combinant la recherche de motifs GraphFrame avec des filtres sur le résultat utilisant des opérations de séquence, agissant sur les colonnes du DataFrame.

Exemple : nous souhaitions identifier les chaînes de 4 sommets a->b->c->d (donc 3 arêtes) vérifiant une certaine propriété définie par une séquence de fonctions. Le processus sera le suivant :
<br> 1. Initialiser l'état sur le chemin.
<br> 2. Mettre à jour l'état en fonction du sommet a.
<br> 3. Mettre à jour l'état en fonction du sommet b.
<br> 4. Mettre à jour l'état en fonction du sommet c.
<br> 5. Et enfin, mettre à jour l'état en fonction du sommet d.

Si l'état final correspond à nos conditions, alors le filtre accepte la chaîne.

Identifier les chaînes de 4 sommets où au moins 2 des 3 arêtes sont des relations "friend". On suivra l'état suivant : nombre actuel d'arêtes "friend".
Ne pas oublier qu'il est possible d'utiliser les fonctions de F (functions) importé en début de notebook.

In [32]:
from pyspark.sql.functions import lit, when

chain_df = g.find("(a)-[ab]->(b); (b)-[bc]->(c); (c)-[cd]->(d)")
chain_df.toPandas()

edge_cols = ["ab", "bc", "cd"]
def count_friends(acc, edge_col):
    return when(col(edge_col)["relationship"] == "friend", acc + 1).otherwise(acc)

chain_df.withColumn(
    "num_friends", reduce(count_friends, edge_cols, lit(0))
).filter(
    col("num_friends") >= 2
).toPandas()

,a,ab,b,bc,c,cd,d,num_friends
0,"(a, Alice, 34)","(a, e, friend)","(e, Esther, 32)","(e, d, friend)","(d, David, 29)","(d, a, friend)","(a, Alice, 34)",3
1,"(e, Esther, 32)","(e, d, friend)","(d, David, 29)","(d, a, friend)","(a, Alice, 34)","(a, b, friend)","(b, Bob, 36)",3
2,"(d, David, 29)","(d, a, friend)","(a, Alice, 34)","(a, b, friend)","(b, Bob, 36)","(b, c, follow)","(c, Charlie, 30)",2
3,"(d, David, 29)","(d, a, friend)","(a, Alice, 34)","(a, e, friend)","(e, Esther, 32)","(e, d, friend)","(d, David, 29)",3
4,"(e, Esther, 32)","(e, d, friend)","(d, David, 29)","(d, a, friend)","(a, Alice, 34)","(a, e, friend)","(e, Esther, 32)",3
5,"(d, David, 29)","(d, a, friend)","(a, Alice, 34)","(a, e, friend)","(e, Esther, 32)","(e, f, follow)","(f, Fanny, 36)",2



## Sous-graphes
GraphFrames fournit une API pour construire des sous-graphes en filtrant sur les arêtes et les sommets.
<br>A partir de notre graphe complet, construire le graphe n'incluant que les personnes de plus de 30 ans et qui ont des amis de plus de 30 ans.
<br>S'aider de la documentation pour filtrer les sommets et les arêtes. Indication : il existe une méthode pour supprimer les Objets isolés une fois les sommets et arêtes filtrés.

In [ ]:
# Votre code ici


## Algorithmes de graphes classiques

GraphFrames fournit un certain nombre d'algorithmes "built-in", dont les plus notables sont :
* Breadth-first search (BFS)
* Connected components
* Strongly connected components
* Label Propagation Algorithm (LPA)
* PageRank (classique et personnalisé)
* Shortest paths
* Triangle count

Dans cette formation, nous nous intéresserons à PageRank et Shortest paths.

## PageRank

Le but de PageRank est d'identifier les sommets "importants".

De votre point de vue, quel sommet du graphe est le plus important ?
<br>Lancer l'algorithme PageRank sur notre graphe avec les paramètres suivants :
<br>resetProbability=0.15
<br>tol=0.01
<br>Afficher les résultats.

In [35]:
g.pageRank(resetProbability=0.15, tol=0.01).vertices.toPandas()

,id,name,age,pagerank
0,a,Alice,34,0.449106
1,b,Bob,36,2.655508
2,c,Charlie,30,2.687830
3,d,David,29,0.328361
4,e,Esther,32,0.370852
5,f,Fanny,36,0.328361
6,g,Gabby,60,0.179982


In [36]:
g.pageRank(resetProbability=0.15, tol=0.01).edges.toPandas()

,src,dst,relationship,weight
0,a,b,friend,0.5
1,a,e,friend,0.5
2,b,c,follow,1.0
3,c,b,follow,1.0
4,d,a,friend,1.0
5,e,d,friend,0.5
6,e,f,follow,0.5
7,f,c,follow,1.0


Contrairement au PageRank standard, qui calcule les scores de pertinence en fonction de la structure globale du graphe, le PageRank personnalisé prend en compte les préférences ou les priorités spécifiques de l'utilisateur.
<br> Dans ce contexte de PageRank personnalisé, sourceId permet de régler le noeud (via son identifiant) à partir duquel l'algorithme commence à évaluer la pertinence des autres nœuds du graphe.
<br> Relancer l'algorithme PageRank en choisissant le noeud "f" correspondant à Fanny comme sourceId.

In [ ]:
# Votre code ici

Essayer d'expliquer le résultat.
A votre avis, quelle(s) propriété(s) de graphe(s) rendent le point de départ d'autant plus important ?


## Shortest paths

Calcule les chemins les plus courts vers un ensemble donné de sommets "repères" (landmarks en anglais).

A l'aide de l'algorithme shortestPaths, calculer , pour chaque sommet, la longueur du plus court chemin reliant ce sommet au sommet "a" et celle du plus court chemin reliant ce sommet au sommet "d" (s'il existe de tels chemins).

In [37]:
g.shortestPaths(landmarks=["a", "d"]).toPandas()

,id,name,age,distances
0,a,Alice,34,"{'d': 2, 'a': 0}"
1,b,Bob,36,{}
2,c,Charlie,30,{}
3,d,David,29,"{'d': 0, 'a': 1}"
4,e,Esther,32,"{'d': 1, 'a': 2}"
5,f,Fanny,36,{}
6,g,Gabby,60,{}


Bonus : Choisir un algorithme supplémentaire dans la liste des algorithmes classiques et le tester !

In [ ]:
# Votre code ici